In [1]:
# Notebook: 01_power_type_i_metric_i
# Monte Carlo operating characteristics of Metric I (directional separations).
# Under a data-generating process fixed to the disclosed band-by-direction default
# rates, this notebook estimates (i) the statistical power to flag value-adding and
# value-destroying adjustment arms as a function of per-cell sample size, and (ii)
# the Type I error of the one-sided two-proportion test used by the decision rule.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RNG = np.random.default_rng(20260101)
BASE = os.path.join("..")
FIG = os.path.join(BASE, "results", "figures")
TAB = os.path.join(BASE, "results", "tables")

# --- grayscale house style; no figure captions (captions live in the paper) ---
sns.set_theme(style="whitegrid", context="paper")
GRAYS = ["0.10", "0.38", "0.62", "0.80"]
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.30",
                     "axes.linewidth": 0.8, "grid.color": "0.88",
                     "grid.linewidth": 0.6, "savefig.bbox": "tight",
                     "figure.dpi": 150})


def save_fig(fig, name):
    # save png and pdf at 600 dpi; paths are deliberately not printed (anonymity)
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, name + "." + ext), dpi=600)
    plt.close(fig)


# --- load the DGP: disclosed band-by-direction default rates (paper Table 4) ---
rates = pd.read_csv(os.path.join(BASE, "data", "disclosed_rates.csv"))
rates["p"] = rates["default_rate_pct"] / 100.0
P = {(r.band, r.direction): (np.nan if pd.isna(r.p) else float(r.p))
     for r in rates.itertuples()}
BANDS = ["BB+", "BB", "BB-", "B+", "B", "B-"]


# --- one-sided two-proportion z-test on a separation (vectorised over replications) ---
def two_prop_p(k1, n1, k0, n0, side):
    # side = +1 tests H1: p1 > p0 ; side = -1 tests H1: p1 < p0
    p1 = k1 / n1
    p0 = k0 / n0
    pp = (k1 + k0) / (n1 + n0)
    se = np.sqrt(pp * (1.0 - pp) * (1.0 / n1 + 1.0 / n0))
    se = np.where(se == 0, np.nan, se)
    z = (p1 - p0) / se
    pv = stats.norm.sf(side * z)
    return np.where(np.isnan(se), 1.0, pv)


# --- analytic minimum cell size for 80% power at a one-sided 5% level ---
def analytic_n(pH, pD):
    za, zb = stats.norm.ppf(0.95), stats.norm.ppf(0.80)
    return (za + zb) ** 2 * (pD * (1 - pD) + pH * (1 - pH)) / (pD - pH) ** 2


# Arms tracked: six downgrade arms (true separation > 0) and the two deficient
# upgrade arms at B+ and B (true separation < 0, value-destroying).
tracked = [("down " + g, g, "D", "H", +1) for g in BANDS] + \
          [("up " + g, g, "U", "H", -1) for g in ("B+", "B")]

N_GRID = [250, 500, 1000, 2000, 5000, 10000, 20000, 40000]
R = 4000
ALPHA = 0.05

power_rows = []
for n in N_GRID:
    draws = {}
    for g in BANDS:
        for d in ("U", "H", "D"):
            p = P[(g, d)]
            draws[(g, d)] = None if np.isnan(p) else RNG.binomial(n, p, size=R)
    for (lab, g, d1, d0, side) in tracked:
        k1, k0 = draws[(g, d1)], draws[(g, d0)]
        if k1 is None or k0 is None:
            continue
        pv = two_prop_p(k1, n, k0, n, side)
        power_rows.append({"arm": lab, "n_per_cell": n,
                           "power": float(np.mean(pv < ALPHA))})
power_df = pd.DataFrame(power_rows)

# --- Type I error: null DGP with p1 == p0 (both cells at the band hold rate) ---
typeI_rows = []
for n in N_GRID:
    fp = []
    for g in BANDS:
        p0 = P[(g, "H")]
        a = RNG.binomial(n, p0, size=R)
        b = RNG.binomial(n, p0, size=R)
        pv_up = two_prop_p(a, n, b, n, +1)   # spurious value-adding flag
        fp.append(np.mean(pv_up < ALPHA))
    typeI_rows.append({"n_per_cell": n, "type_I_error": float(np.mean(fp))})
typeI_df = pd.DataFrame(typeI_rows)

# --- empirical minimum n for 80% power vs analytic prediction, per tracked arm ---
minn_rows = []
for (lab, g, d1, d0, side) in tracked:
    sub = power_df[power_df["arm"] == lab].sort_values("n_per_cell")
    n_emp = np.nan
    xs, ys = sub["n_per_cell"].values, sub["power"].values
    for i in range(len(xs) - 1):
        if ys[i] < 0.80 <= ys[i + 1]:
            n_emp = xs[i] + (0.80 - ys[i]) * (xs[i + 1] - xs[i]) / (ys[i + 1] - ys[i])
            break
    if ys[0] >= 0.80:
        n_emp = xs[0]
    pH, pOther = P[(g, d0)], P[(g, d1)]
    hi, lo = max(pH, pOther), min(pH, pOther)
    minn_rows.append({"arm": lab,
                      "true_sep_pp": round((pOther - pH) * 100, 2),
                      "n_empirical_80pct": (None if np.isnan(n_emp) else int(round(n_emp))),
                      "n_analytic_80pct": int(round(analytic_n(lo, hi)))})
minn_df = pd.DataFrame(minn_rows)

# --- save tables (no path echo) ---
power_df.to_csv(os.path.join(TAB, "t01_power_by_arm.csv"), index=False)
typeI_df.to_csv(os.path.join(TAB, "t01_type_i_error.csv"), index=False)
minn_df.to_csv(os.path.join(TAB, "t01_min_n_80pct.csv"), index=False)

# --- figure 1: power curves for representative arms (grayscale, linestyle-coded) ---
rep = [("down B-", "0.10", "-", "o"),
       ("down B+", "0.10", "--", "s"),
       ("down BB", "0.45", "-.", "^"),
       ("up B+", "0.45", ":", "D")]
fig, ax = plt.subplots(figsize=(6.2, 4.2))
for lab, c, ls, mk in rep:
    sub = power_df[power_df["arm"] == lab].sort_values("n_per_cell")
    ax.plot(sub["n_per_cell"], sub["power"], color=c, linestyle=ls,
            marker=mk, markersize=5, markerfacecolor="white",
            markeredgecolor=c, linewidth=1.4, label=lab)
ax.axhline(0.80, color="0.55", linewidth=0.9, linestyle=(0, (1, 1)))
ax.set_xscale("log")
ax.set_xlabel("Firms per cell (log scale)")
ax.set_ylabel("Power at one-sided 5% level")
ax.set_ylim(0, 1.02)
ax.legend(frameon=False, loc="lower right", fontsize=9)
save_fig(fig, "fig01_power_curves")

# --- figure 2: Type I error vs nominal level (grayscale) ---
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.plot(typeI_df["n_per_cell"], typeI_df["type_I_error"], color="0.10",
        marker="o", markersize=5, markerfacecolor="white",
        markeredgecolor="0.10", linewidth=1.4)
ax.axhline(ALPHA, color="0.55", linewidth=0.9, linestyle=(0, (1, 1)))
ax.set_xscale("log")
ax.set_xlabel("Firms per cell (log scale)")
ax.set_ylabel("Empirical Type I error")
ax.set_ylim(0, 0.10)
save_fig(fig, "fig02_type_i_error")

# --- compact on-screen summary (no file paths) ---
print("Monte Carlo replications per point:", R)
print("\nType I error (should sit near 0.05):")
print(typeI_df.to_string(index=False))
print("\nMinimum firms per cell for 80% power (empirical vs analytic):")
print(minn_df.to_string(index=False))


Monte Carlo replications per point: 4000

Type I error (should sit near 0.05):
 n_per_cell  type_I_error
        250      0.042708
        500      0.049125
       1000      0.050625
       2000      0.049417
       5000      0.049833
      10000      0.045083
      20000      0.051625
      40000      0.049500

Minimum firms per cell for 80% power (empirical vs analytic):
     arm  true_sep_pp  n_empirical_80pct  n_analytic_80pct
down BB+         0.66             1754.0              1746
 down BB         0.11                NaN             68510
down BB-         0.40             9093.0              8933
 down B+         1.24             1641.0              1534
  down B         1.97              928.0               914
 down B-         2.77              757.0               706
   up B+         0.25                NaN             28364
    up B         0.02                NaN           6029224
